# Triple-Barrier Parameter AnalysisThis notebook provides a workflow for analyzing and visualizing the choice of parameters for the Triple-Barrier labeling method.The process is as follows:1. **Load Execution Price Data**: Fetch the realistic bid/ask execution prices for a given market and date range.2. **Calculate & Visualize Volatility**: Compute a rolling volatility measure from the historical data, which will be used to set dynamic barriers.3. **Define Barriers & Generate Labels**: Use the calculated volatility to define the triple barriers and generate the resulting labels.4. **Analyze Label Distribution**: Summarize and plot the distribution of outcomes (Win, Loss, Timeout) to understand the impact of the chosen parameters.

## SetupImport libraries and define the core parameters for the analysis.

In [ ]:
import syssys.path.insert(0, '../src')import pandas as pdimport numpy as npimport matplotlib.pyplot as pltimport seaborn as snsfrom qate.core.model import Market, Exchange, Symbolfrom qate.store.parquet_store import ParquetStorefrom qate.util.dt_range import DtRange# --- Display and Plotting Settings ---pd.set_option('display.max_columns', 50)pd.set_option('display.max_rows', 100)sns.set_style("whitegrid")%matplotlib inline# --- Core Analysis Parameters ---PARQUET_STORE_ROOT_DIR = "../data/parquet" # Adjust this path as neededMARKET = Market(Exchange.BINANCE, Symbol.BTC_USDT) # Example marketEXECUTION_SIZE = "1.0" # The order size for which to get execution prices# --- Date Range for Analysis ---DATA_START_DATE = "2023-01-03"DATA_END_DATE = "2023-01-04" # Analyze a single day of dataDATA_DT_RANGE = DtRange(DATA_START_DATE, DATA_END_DATE)# --- Triple-Barrier Hyperparameters ---VOLATILITY_LOOKBACK_MINUTES = 60 # Use a 1-hour window for rolling volatilityBARRIER_MULTIPLIER = 2.0 # Set barriers at 2.0x the rolling volatilityTIMEOUT_SEC = 45 # Set a 45-second timeout for the vertical barrierTIMEOUT_DELTA = pd.Timedelta(seconds=TIMEOUT_SEC)print("Setup complete. Parameters are defined.")

## 1. Load Execution Price DataLoad the execution price data which contains the realistic bid, ask, and mid prices for our chosen market and order size.

In [ ]:
full_parquet_store = ParquetStore(PARQUET_STORE_ROOT_DIR)# We need to read a slightly larger range to ensure the volatility window is populatedvol_read_start_dt = pd.to_datetime(DATA_START_DATE) - pd.Timedelta(minutes=VOLATILITY_LOOKBACK_MINUTES)load_range = DtRange(vol_read_start_dt.strftime('%Y-%m-%d %H:%M:%S'), DATA_END_DATE)print(f"Loading execution prices for {MARKET.symbol.name} (size: {EXECUTION_SIZE})...")price_store = full_parquet_store.create_execution_price(MARKET, execution_size_str=EXECUTION_SIZE)df = price_store.read(load_range)if 'ts' in df.columns:    df.index = pd.to_datetime(df['ts'], unit='s')    df = df.drop(columns=['ts'])print(f"Loaded {len(df)} data points from {df.index.min()} to {df.index.max()}.")df.head()

## 2. Calculate & Visualize VolatilityHere we implement **Methodology 1**: calculating dynamic barriers based on recent volatility. We'll compute the rolling standard deviation of 1-minute returns.

In [ ]:
# 1. Resample mid-price to 1-minute bars to calculate returnsone_min_mid_price = df['mid'].resample('1min').last()# 2. Calculate 1-minute percentage returnsone_min_returns = one_min_mid_price.pct_change()# 3. Calculate the rolling standard deviation (volatility)# This gives us a volatility estimate at a 1-minute frequencyrolling_vol_1min = one_min_returns.rolling(window=VOLATILITY_LOOKBACK_MINUTES).std()# 4. Align the volatility back to the original tick-level DataFrame# We forward-fill to apply the last known volatility to each tick within that minute.df['volatility'] = rolling_vol_1min.reindex(df.index, method='ffill')# Clean up data to be within the original desired date rangedf = df.loc[DATA_DT_RANGE.start_dt:DATA_DT_RANGE.end_dt].dropna()print(f"Calculated rolling volatility. {len(df)} data points remaining after alignment.")df[['mid', 'volatility']].head()

In [ ]:
# Plot the mid-price and the calculated volatilityfig, ax1 = plt.subplots(figsize=(15, 7))color = 'tab:blue'ax1.set_xlabel('Time')ax1.set_ylabel('Mid Price', color=color)ax1.plot(df.index, df['mid'], color=color, label='Mid Price')ax1.tick_params(axis='y', labelcolor=color)ax2 = ax1.twinx()  # instantiate a second axes that shares the same x-axiscolor = 'tab:red'ax2.set_ylabel('1-Hour Rolling Volatility', color=color)ax2.plot(df.index, df['volatility'], color=color, alpha=0.7, label='Volatility')ax2.tick_params(axis='y', labelcolor=color)plt.title(f'{MARKET.symbol.name} Mid Price and Rolling Volatility')fig.tight_layout()plt.show()

## 3. Define Barriers & Generate LabelsNow we use the calculated `volatility` series to define our dynamic barriers and generate labels. For simplicity, we will only generate labels for a **long scenario**.

In [ ]:
# Define the profit-take and stop-loss levels for each timestamp dynamicallydf['pt_pct'] = BARRIER_MULTIPLIER * df['volatility']df['sl_pct'] = BARRIER_MULTIPLIER * df['volatility']print("Profit-take and Stop-loss levels (in %):")(df[['pt_pct', 'sl_pct']] * 100).describe()

In [ ]:
def get_labels_with_dynamic_barriers(prices_df: pd.DataFrame, timeout_delta: pd.Timedelta) -> pd.DataFrame:    """    Computes triple-barrier labels for a long scenario using dynamic pt/sl levels.    `prices_df` must contain 'ask', 'bid', 'pt_pct', 'sl_pct' columns.    """    n = len(prices_df)    results = []    # Extract series for faster access    entry_ask_prices = prices_df['ask']    future_bid_prices = prices_df['bid']    pt_pct_series = prices_df['pt_pct']    sl_pct_series = prices_df['sl_pct']    for i in range(n):        entry_ts = prices_df.index[i]        entry_price = entry_ask_prices.iloc[i]        timeout_ts = entry_ts + timeout_delta        # Get the specific pt/sl for this entry time        pt_level = pt_pct_series.iloc[i]        sl_level = sl_pct_series.iloc[i]        if pd.isna(entry_price) or pd.isna(pt_level) or pd.isna(sl_level):            continue        upper_barrier = entry_price * (1 + pt_level)        lower_barrier = entry_price * (1 - sl_level)        # Find future path        future_path = future_bid_prices.loc[future_bid_prices.index > entry_ts]                # Find barrier hits        pt_hits = future_path[future_path >= upper_barrier]        sl_hits = future_path[future_path <= lower_barrier]        first_pt_hit_ts = pt_hits.index[0] if not pt_hits.empty else None        first_sl_hit_ts = sl_hits.index[0] if not sl_hits.empty else None                first_hit_ts = min(ts for ts in [first_pt_hit_ts, first_sl_hit_ts] if ts) if any([first_pt_hit_ts, first_sl_hit_ts]) else None                label = 0        if first_hit_ts and first_hit_ts <= timeout_ts:            if future_path[first_hit_ts] >= upper_barrier:                label = 1 # Profit Take            else:                label = -1 # Stop Loss        else:            label = 0 # Timeout        results.append({"label": label})    return pd.DataFrame(results, index=prices_df.index[:len(results)])# Generate the labelsprint("Generating labels... (This may take a few moments)")labels_df = get_labels_with_dynamic_barriers(df, TIMEOUT_DELTA)df = df.join(labels_df)print("Label generation complete.")df[['mid', 'volatility', 'pt_pct', 'sl_pct', 'label']].head()

## 4. Analyze Label DistributionFinally, we analyze the outcomes. How many of our potential trades resulted in a win, a loss, or a timeout? This is critical for understanding if the barrier parameters are well-chosen. If 99% of labels are timeouts, the barriers might be too wide or the timeout too short.

In [ ]:
label_counts = df['label'].value_counts().sort_index()# Create a mapping for human-readable labelslabel_map = {    1: f'Win (PT = {BARRIER_MULTIPLIER}x Vol)',    -1: f'Loss (SL = {BARRIER_MULTIPLIER}x Vol)',    0: f'Timeout ({TIMEOUT_SEC}s)'}label_counts.rename(index=label_map, inplace=True)print("Label Distribution Summary:")summary_df = pd.DataFrame({    'count': label_counts,    'percentage': (label_counts / label_counts.sum() * 100).round(2)})print(summary_df)# Plot the distributionplt.figure(figsize=(10, 6))ax = sns.barplot(x=summary_df.index, y=summary_df['count'])ax.set_title('Triple-Barrier Label Distribution')ax.set_ylabel('Count')ax.set_xlabel('Outcome')# Add percentage labels to the barsfor i, p in enumerate(ax.patches):    height = p.get_height()    ax.text(p.get_x() + p.get_width()/2.,            height + 0.1,            f"{summary_df['percentage'].iloc[i]}%",            ha="center", va="bottom")plt.xticks(rotation=15)plt.tight_layout()plt.show()